<p align="center">
  <img src=".github/images/banners/setup.png" alt="Setup Foundry Project: one-time setup before running release capsules" width="900" />
</p>

## Pre-Requisites

- An Azure subscription with access to Microsoft Foundry 
- Familiarity with Python, VS Code and GitHub Codespaces

Codespaces has everything else. We do this once; every capsule reuses the result.


## What We Will Do

1. Run the setup script to create a Foundry project — if we haven't already
2. Verify we're signed in to Azure with working credentials
3. Verify the script created a `.env` file for us
4. Verify the `.env` has the basic variables populated

> 💡 Run each cell with **Shift + Enter**. Before each one, take a guess at what it will print.

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 1. Do we have a Foundry project yet?

If `.env` already points at a project, we skip this step. Otherwise, fill in the values below and the [setup script](1-scripts/setenv.sh) creates the project (or connects an existing one) and writes `.env`. Not signed in yet? Run `az login --use-device-code` in a terminal first.

In [1]:
MODE = "create"           # ✏️ "create" a new project, or "use" one we already have
RESOURCE_GROUP = ""       # ✏️ e.g. "rg-model-releases"
PROJECT_NAME = ""         # ✏️ e.g. "model-releases"
LOCATION = "eastus2"      # ✏️ only used when MODE == "create"
OVERWRITE = False         # ✏️ True replaces an existing .env

In [2]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv

ENV_FILE = Path(".env")
load_dotenv(ENV_FILE, override=True)

if os.getenv("MICROSOFT_FOUNDRY_ENDPOINT") and not OVERWRITE:
    print("⏭️  .env already points at a Foundry project. Skipping — set OVERWRITE = True to redo it.")
else:
    if MODE not in ("use", "create"):
        raise SystemExit('MODE must be "use" or "create".')
    if not RESOURCE_GROUP or not PROJECT_NAME:
        raise SystemExit("Fill in RESOURCE_GROUP and PROJECT_NAME in the cell above, then run both cells again.")
    args = ["bash", "1-scripts/setenv.sh", f"--{MODE}", RESOURCE_GROUP, PROJECT_NAME]
    if MODE == "create":
        args.append(LOCATION)
    if OVERWRITE:
        args.append("--force")
    subprocess.run(args, check=True)

SystemExit: Fill in RESOURCE_GROUP and PROJECT_NAME in the cell above, then run both cells again.

/home/vscode/.local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 2. Are we signed in to Azure with working credentials?

Capsules authenticate with our Azure identity. We confirm the CLI sign-in, then ask for a token the same way the capsules do.

> 🔮 **Guess first:** will it print the subscription we expect, or a different default?

In [ ]:
import json

from azure.identity import DefaultAzureCredential

try:
    account = subprocess.run(["az", "account", "show", "--output", "json"], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit("Azure CLI not found. Open this repo in Codespaces, or install it: https://aka.ms/installazurecli")
if account.returncode != 0:
    raise SystemExit("Not signed in. Run `az login --use-device-code` in a terminal, then re-run this cell.")
print(f"✅ Signed in to: {json.loads(account.stdout)['name']}")

DefaultAzureCredential().get_token("https://cognitiveservices.azure.com/.default")
print("✅ Credentials work: got a token for Microsoft Foundry")

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 3. Did the script create our `.env`?

`.env` lives at the repo root and Git ignores it, so our key stays local. We list the variable names it holds — never the values.

> 🔮 **Guess first:** how many variables will it hold — just the project details, or more?

In [ ]:
if not ENV_FILE.exists():
    raise SystemExit("No .env yet. Run step 1, then come back here.")
names = [line.split("=", 1)[0] for line in ENV_FILE.read_text().splitlines() if "=" in line and not line.startswith("#")]
print(f"✅ Found {ENV_FILE.resolve()} with {len(names)} variables:")
print("   " + ", ".join(names))

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 4. Are the basic variables populated?

Every capsule starts with this same check. When it prints `Ready`, any capsule can find our project.

In [ ]:
load_dotenv(ENV_FILE, override=True)


def check_env(required):
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise EnvironmentError(f"Missing {missing} in .env. Re-run step 1 with OVERWRITE = True, or add them by hand.")
    return "✅ Ready: " + os.environ["MICROSOFT_FOUNDRY_ENDPOINT"]


print(check_env(["MICROSOFT_FOUNDRY_ENDPOINT", "MICROSOFT_FOUNDRY_API_KEY"]))

### 🔧 Break it on purpose

What happens when a capsule needs a setting we don't have yet?

> 🔮 **Guess first:** will the message name the missing setting, or just say something failed?

In [ ]:
try:
    check_env(["MICROSOFT_FOUNDRY_ENDPOINT", "AZURE_NOT_A_REAL_DEPLOYMENT"])
except EnvironmentError as err:
    print("🔧", err)

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## What We Did

1. ✅ Ran the setup script — or confirmed we already had a Foundry project
2. ✅ Verified our Azure sign-in and credentials
3. ✅ Verified the script created `.env`
4. ✅ Verified the basic variables are populated — and saw how a capsule names a missing one

➡️ **Next:** pick a capsule from [What's new?](README.md#whats-new) and run it top to bottom. Its first cells list any models to deploy — in the project, open **Model catalog → Deploy**.

Stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

**Learn more:** [Get started with Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code) · [Create a project](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects)